# zkp lesson 1: zkp of discrete logs

## motivation

suppose you want to prove to someone you know some secret value without revealing that value to that person. we can use the fact that the discrete logarithm problem is believed to be difficult to do this.

## identities

in this system, identities are made up of four values. the first is a secret value `x`. the next three are all public:

* a large prime, `p`

* a multiplicate generator of the group ZZ/p, `g`

* a group element `A = g^x (mod p)`

In [107]:
# TODO(kkl): What is the proper name for these parts of the protocol?

# global system paramters
p = next_prime(2^16)
Zp = Zmod(p)
g = Zp.multiplicative_generator()

class Verifier(object):
    
    def accept_commitment(self, public, h):
        self.public = public
        self.h = h
        self.b = randint(0, 1)
        return self.b
    
    def verify_proof(self, s):
        return g^s == self.h*self.public^self.b
    
class Prover(object):
    
    def __init__(self):
        self.private = randint(1, p)
        self.public = g^self.private
    
    def generate_commitment(self):
        self.r = randint(1, p)
        return g^self.r
    
    def generate_proof(self, b):
        return Zmod(p-1)(self.r + b*self.private)

prover = Prover()
verifier = Verifier()

# step 1: prover generates a commitment
# commitment = g^r s.t. r = random integer
commitment = prover.generate_commitment()

# step 2: the verifier generates a random bit after getting a commitment
random_bit = verifier.accept_commitment(prover.public, commitment)

# step 3: the prover generates it's proof of either r or the private key
# proof = r + random_bit*private_key
proof = prover.generate_proof(random_bit)

# step 4: verifier verifies the proof
# assert g^s == h*public^random_bit
assert(verifier.verify_proof(proof))

### why?... do you take s mod p-1?

fermats little thereom: a^(p-1) = 1 (mod p) where p is prime.

`r + b*x mod p-1` can be written as `r + b*x + k*(p-1)` for some `k`. 

`g^s = g^(r + b*x + k*(p-1)) = g^r * g^(b*x) * g^(k*(p-1)) (mod p)`

`g^(k*(p-1)) = (g^p-1)^k (mod p)` which is congruent to 1 by flt.

### why?... do you even choose a random bit b?

the verifier can cheat in both the b=0 and b=1 case!

in the `b=0` case, an attacker "prover" can pass the verification process without knowledge of `x`, but he has to first commit that he knows the discrete log of `h` (i.e. the value `r` above). however, the attacker would be caught without knowledge of `x` if `b=1`.

in the `b=1` case, an attacker "prover" can pass verification by submitting an `h` value of `g^s * public^-1` for some random `s`.
this would pass:

`g^s == h*public^b` because
`g^s == (g^s * public^-1)*public`
`g^s == g^s`

however, in this case, the attacker "prover" doesn't know the discrete log of this malicious `h` value.

in other words, in both cases, the attacker has to commit to either knowing the discrete log of `h` or not. knowing the discrete log for `h` helps for `b=0` but won't for `b=1`.

### attack!

if the value `r` is re-used (or you can just re-run the protocol from the middle of a proof) the verifier can cause the prover to leak `x`.

if on run 1, the verifier sends `b=1`, the prover sends `s = r+x`.

if on run 2, the verifier sends `b=0` with `r` the same as run 1, the prover sends `s' = r`. 

the verifier can then calculate `s' - s = r - r + x = x`.

### attack!

biased `b` value increases the attackers odds of winning!

### attack!

limited runs of the protocol increases the attackers odds of winning!

# zkp lesson 2: turning a zkp into a signature

Right now the above proof is interactive. Meaning, the prover and verifier both have to contribute information in a specific order.

The verifier cannot just provide random `b` values to the prover before the prover commits, as the prover can just cheat knowing the `b` value.

So the entity generating the siganture (the signer) needs to be both the prover and the verifier. In order to prevent the signer from generating `b` values as needed, we replace a single bit `b` with the output of a hash function.


In [120]:
import hashlib

def to_int(ascii_string):
  base, total = 1, 0
  for char in ascii_string:
    total += ord(char) * base
    base *= 10
  return total


def digest(mx, mr, ar):
    md5 = hashlib.md5()
    md5.update(str([mx, mr, ar]))
    return int(md5.hexdigest(), 16)

def encode(private, message):
    r = randint(1, p)
    m = Zp(to_int(message))
    mx = m^private
    mr = m^r
    ar = g^r
    c = digest(mx, mr, ar)
    return r, mx, mr, ar, c        
   
def verify(public_key, message, signature):
    c = digest(signature.mx, signature.mr, signature.ar)
    m = Zp(to_int(message))
    return all([
        g^signature.sig == public_key^c * signature.ar,
        m^signature.sig == signature.mx^c * signature.mr
    ])

class Signature(object):
    
    def __init__(self, mx, mr, ar, sig):
        self.mx = mx
        self.mr = mr
        self.ar = ar
        self.sig = sig
                  
class Signer(Prover):

    def sign(self, message):
        r, mx, mr, ar, c = encode(self.private, message)
        sig = Zmod(p-1)(c*self.private + r)
        return Signature(mx, mr, ar, sig)

In [121]:
signer = Signer()
signature = signer.sign("foo")
print(verify(signer.public, "foo", signature))

True
